In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/hotels.csv")

df.head

<bound method NDFrame.head of                hotel  is_canceled  lead_time  arrival_date_year  \
0       Resort Hotel            0        342               2015   
1       Resort Hotel            0        737               2015   
2       Resort Hotel            0          7               2015   
3       Resort Hotel            0         13               2015   
4       Resort Hotel            0         14               2015   
...              ...          ...        ...                ...   
119385    City Hotel            0         23               2017   
119386    City Hotel            0        102               2017   
119387    City Hotel            0         34               2017   
119388    City Hotel            0        109               2017   
119389    City Hotel            0        205               2017   

       arrival_date_month  arrival_date_week_number  \
0                    July                        27   
1                    July                        27   


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

RAW_PATH = Path("../data/raw/hotels.csv")
INTERIM_DIR = Path("data/interim")
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = INTERIM_DIR / "hotels_cleaned.parquet"

MONTH_ORDER = {
    "January": 1, "February": 2, "March": 3, "April": 4,
    "May": 5, "June": 6, "July": 7, "August": 8,
    "September": 9, "October": 10, "November": 11, "December": 12
}

EXPECTED_COLUMNS = [
    "hotel", "is_canceled", "lead_time", "arrival_date_year", "arrival_date_month",
    "arrival_date_week_number", "arrival_date_day_of_month", "stays_in_weekend_nights",
    "stays_in_week_nights", "adults", "children", "babies", "meal", "country",
    "market_segment", "distribution_channel", "is_repeated_guest",
    "previous_cancellations", "previous_bookings_not_canceled", "reserved_room_type",
    "assigned_room_type", "booking_changes", "deposit_type", "agent", "company",
    "days_in_waiting_list", "customer_type", "adr", "required_car_parking_spaces",
    "total_of_special_requests", "reservation_status", "reservation_status_date"
]

def main():
    df = pd.read_csv(RAW_PATH)

    missing = [c for c in EXPECTED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing expected columns: {missing}")

    # Normalize blank text to NaN
    for col in [
        "meal", "country", "market_segment", "distribution_channel",
        "deposit_type", "customer_type", "agent", "company",
        "reserved_room_type", "assigned_room_type", "reservation_status"
    ]:
        df[col] = df[col].replace(r"^\s*$", np.nan, regex=True)

    # Safe numeric conversion
    numeric_cols = [
        "is_canceled", "lead_time", "arrival_date_year", "arrival_date_week_number",
        "arrival_date_day_of_month", "stays_in_weekend_nights", "stays_in_week_nights",
        "adults", "children", "babies", "is_repeated_guest",
        "previous_cancellations", "previous_bookings_not_canceled",
        "booking_changes", "days_in_waiting_list", "adr",
        "required_car_parking_spaces", "total_of_special_requests"
    ]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Fill children for guest-count logic
    df["children"] = df["children"].fillna(0)

    # Build arrival date from year/month/day
    df["arrival_month_num"] = df["arrival_date_month"].map(MONTH_ORDER)
    
    df["arrival_date"] = pd.to_datetime(
        dict(
            year=df["arrival_date_year"],
            month=df["arrival_month_num"],
            day=df["arrival_date_day_of_month"]
        ),
        errors="coerce"
    )

    # Parse reservation status date
    df["reservation_status_date"] = pd.to_datetime(df["reservation_status_date"], errors="coerce")

    # QA flags
    df["flag_missing_country"] = df["country"].isna().astype(int)
    df["flag_missing_agent"] = df["agent"].isna().astype(int)
    df["flag_missing_company"] = df["company"].isna().astype(int)
    df["flag_bad_arrival_date"] = df["arrival_date"].isna().astype(int)

    # Useful standardizations
    df["meal"] = df["meal"].replace({"Undefined": "SC"})
    df["country"] = df["country"].fillna("Unknown")
    df["agent"] = df["agent"].fillna("Unknown")
    df["company"] = df["company"].fillna("Unknown")

    # Remove exact duplicates only
    df = df.drop_duplicates()

    df.to_parquet(OUTPUT_PATH, index=False)
    print(f"Saved cleaned data to {OUTPUT_PATH}")
    print(df.shape)

if __name__ == "__main__":
    main()


ArrowTypeError: ("Expected bytes, got a 'float' object", 'Conversion failed for column agent with type object')

In [2]:
import os

print(os.getcwd())
print(os.listdir())

/workspaces/Hotel-Customer-Insights/notebooks
['01_data_audit.ipynb', 'data']


In [4]:
print(os.listdir("../data/raw"))

['hotels.csv']
